In [17]:
print("Training process for Random Forest...\n\n🛢️ Random Forest Pipeline: SMOTE -> Random Forest")
rf_pipeline = ImbPipeline([
    (
        "smote", 
        SMOTE(random_state=42)
    ),
    (
        "model",
        RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        )
    )
])
print("✅ Done!\n")

print("🔎 Random Forest hyperparameter search...")
rf_params = {
    "model__n_estimators": randint(150, 700),
    "model__max_depth": [10, 15, 20, 25, 30, None],
    "model__min_samples_split": randint(5, 50),
    "model__min_samples_leaf": randint(2, 15),
    "model__max_features": ["sqrt", "log2", 0.3, 0.5]
}
print("✅ Done!\n")

print("🚀 Initializing cross-validation object...")
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
print("✅ Done!\n")

print("🎲 Initializing a search manager object...")
rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_params,
    n_iter = 30,
    scoring="roc_auc",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)
print("✅ Done!\n")

Training process for Random Forest...

🛢️ Random Forest Pipeline: SMOTE -> Random Forest
✅ Done!

🔎 Random Forest hyperparameter search...
✅ Done!

🚀 Initializing cross-validation object...
✅ Done!

🎲 Initializing a search manager object...
✅ Done!



In [18]:
print("🎓 Fitting the search manager...")
rf_search.fit(X_train_rf, y_train_rf)

print(f"Best hyperparameters for Random Forest model are:\n{rf_search.best_params_}")
print(f"The best score is:{rf_search.best_score_}\n")

best_rf = rf_search.best_estimator_
print("💾 Saving best performing Random Forest model...")

try:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    joblib.dump(best_rf, f"../models/random_forest_best_{timestamp}.joblib")
    joblib.dump(rf_search, f"../models/rf_random_search_{timestamp}.joblib")
    print("✅ Random Forest model with best performance score was successfully saved!")
    
except:
    print("❌ Something went wrong during the model serialization...Check Traceback call")

🎓 Fitting the search manager...
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best hyperparameters for Random Forest model are:
{'model__max_depth': 20, 'model__max_features': 0.3, 'model__min_samples_leaf': 2, 'model__min_samples_split': 12, 'model__n_estimators': 568}
The best score is:0.9963681689443057

💾 Saving best performing Random Forest model...
✅ Random Forest model with best performance score was successfully saved!
[CV] END model__max_depth=30, model__max_features=0.5, model__min_samples_leaf=9, model__min_samples_split=28, model__n_estimators=280; total time= 4.3min
[CV] END model__max_depth=None, model__max_features=0.3, model__min_samples_leaf=4, model__min_samples_split=48, model__n_estimators=393; total time= 4.0min
[CV] END model__max_depth=None, model__max_features=sqrt, model__min_samples_leaf=5, model__min_samples_split=44, model__n_estimators=537; total time= 1.5min
[CV] END model__max_depth=15, model__max_features=log2, model__min_samples_leaf=13,

In [65]:
from sklearn.ensemble import IsolationForest

print("Train Isolation Forest on training period (time_step 1-34)\n")
if_model = IsolationForest(
    n_estimators=300,
    max_samples=2048,
    contamination=0.05,
    max_features=0.5,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)


if_model.fit(X_train_if)


print("Isolation Forest trained")

Train Isolation Forest on training period (time_step 1-34)

Isolation Forest trained


In [66]:
print("Create prediction helper function")
def create_if_predictions(
    model,
    X,
    df
):
    """
    Generate Isolation Forest outputs.
    """
    
    predictions = pd.DataFrame({
        "txId": df["txId"].values,
        "time_step": df["time_step"].values,

        # sklearn score:
        # higher = more normal
        # lower = more abnormal
        "if_score": model.score_samples(X),

        # sklearn decision:
        # negative = anomaly
        # positive = normal
        "decision_function": model.decision_function(X),

        # binary output
        "prediction": (
            model.predict(X) == -1
        ).astype(int)
    })


    return predictions

print("Prediction helper function created!")

Create prediction helper function
Prediction helper function created!


In [67]:
val_predictions_df = create_if_predictions(
    if_model,
    X_val_if,
    validation_if_df
)


val_predictions_df.head()
print("Generated validation predictions")

Generated validation predictions


In [68]:
val_eval_df = val_predictions_df.merge(
    validation_labels_if,
    on="txId",
    how="inner"
)


print(
    "Validation labeled transactions:",
    len(val_eval_df)
)

val_eval_df.head()
print("Created validation evaluation dataset")

Validation labeled transactions: 7829
Created validation evaluation dataset


In [69]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

y_true = val_eval_df["is_illicit"]

y_score = val_eval_df["if_score"]


print("==============================")
print("Validation IF ranking")
print("==============================")


print(
    "ROC-AUC:",
    roc_auc_score(
        y_true,
        y_score
    )
)


print(
    "PR-AUC:",
    average_precision_score(
        y_true,
        y_score
    )
)

Validation IF ranking
ROC-AUC: 0.9070769007755307
PR-AUC: 0.6219264379570125


In [70]:
print("Prepare final training data (time_step 1-41)")
X_train_if_full = (
    unsupervised_full_df[
        unsupervised_full_df["time_step"] <= 41
    ]
    .drop(
        columns=["txId", "time_step"]
    )
)


print(
    "Final training shape:",
    X_train_if_full.shape
)


Prepare final training data (time_step 1-41)
Final training shape: (166945, 165)


In [71]:
final_if_model = IsolationForest(
    n_estimators=300,
    max_samples=2048,
    contamination=0.05,
    max_features=0.5,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)


final_if_model.fit(
    X_train_if_full
)


print("Final IF trained on time steps 1-41")

Final IF trained on time steps 1-41


In [72]:
print("Generate final test predictions")
test_predictions_df = create_if_predictions(
    model=final_if_model,
    X=X_test_if,
    df=test_if_df
)


test_predictions_df.head()

Generate final test predictions


,txId,time_step,if_score,decision_function,prediction
0,72340883,42,-0.413166,0.045693,0
1,73005229,42,-0.596482,-0.137623,1
2,73243693,42,-0.356850,0.102009,0
3,73243691,42,-0.357684,0.101175,0
4,3866934,42,-0.388741,0.070118,0


In [74]:
print("Merge test labels for evaluation\n")
test_eval_df = test_predictions_df.merge(
    test_labels_if,
    on="txId",
    how="inner"
)


print(
    "Test labeled transactions:",
    len(test_eval_df)
)

Merge test labels for evaluation

Test labeled transactions: 8841


In [75]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)


y_true = test_eval_df["is_illicit"]


y_score = test_eval_df["if_score"]


print("==============================")
print("FINAL TEST IF RANKING")
print("==============================")


print(
    "ROC-AUC:",
    roc_auc_score(
        y_true,
        y_score
    )
)


print(
    "PR-AUC:",
    average_precision_score(
        y_true,
        y_score
    )
)

FINAL TEST IF RANKING
ROC-AUC: 0.7191210766293947
PR-AUC: 0.28734625439743194
